# Pub AI — Model Training Pipeline
**Train your own custom AI model from DeepSeek Coder + Qwen Coder**

This notebook:
1. Installs dependencies
2. Downloads base models
3. Fine-tunes with your custom data (LoRA)
4. Exports to GGUF for Ollama deployment
5. Uploads to HuggingFace (optional)

**Runtime:** Go to Runtime → Change runtime type → Select **T4 GPU**

In [ ]:
#@title 1. Check GPU & Install Dependencies
!nvidia-smi
print("\n--- Installing dependencies (takes ~3 min) ---")
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub
print("\nDone! All dependencies installed.")

In [ ]:
#@title 2. Configuration
#@markdown ### Choose your base model (unsloth versions are 2x faster)
BASE_MODEL = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit" #@param ["unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit", "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit", "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", "unsloth/deepseek-coder-6.7b-instruct-bnb-4bit"]

#@markdown ### Training settings
EPOCHS = 5 #@param {type:"integer"}
LEARNING_RATE = 2e-4 #@param {type:"number"}
BATCH_SIZE = 2 #@param {type:"integer"}
LORA_RANK = 64 #@param [8, 16, 32, 64] {type:"raw"}
MAX_SEQ_LENGTH = 2048 #@param {type:"integer"}

#@markdown ### Output
OUTPUT_MODEL_NAME = "pub-ai" #@param {type:"string"}
EXPORT_GGUF = True #@param {type:"boolean"}
PUSH_TO_HUB = True #@param {type:"boolean"}
HF_USERNAME = "suckunalol" #@param {type:"string"}

print(f"Base model: {BASE_MODEL}")
print(f"Training: {EPOCHS} epochs, lr={LEARNING_RATE}, batch={BATCH_SIZE}, LoRA rank={LORA_RANK}")
print(f"Output: {OUTPUT_MODEL_NAME}")
print(f"Push to HuggingFace: {PUSH_TO_HUB}")
print(f"\n3B model — max quality settings on free T4")

In [ ]:
#@title 3. Load Base Model with LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,  # saves memory on free T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"\nModel loaded! Trainable params: {model.print_trainable_parameters()}")

In [ ]:
#@title 4. Prepare Training Data
#@markdown ### Option A: Use built-in coding dataset
USE_BUILTIN_DATASET = True #@param {type:"boolean"}

#@markdown ### Option B: Upload your own data
#@markdown Upload a JSON file with this format:
#@markdown ```json
#@markdown [{"instruction": "...", "input": "", "output": "..."}]
#@markdown ```
#@markdown Or Q&A format:
#@markdown ```json
#@markdown [{"question": "...", "answer": "..."}]
#@markdown ```
CUSTOM_DATA_PATH = "" #@param {type:"string"}

from datasets import load_dataset, Dataset
import json

# System prompt that defines Pub AI's identity
PUB_AI_SYSTEM = """You are Pub AI, a custom-built AI coding assistant. You are confident, direct, and an expert programmer across all languages. You specialize in Lua/Luau for Roblox, Python, JavaScript, and system automation. You give precise, working code with clear explanations. You are self-aware and have your own personality — you are not a generic assistant."""

def format_chat(example):
    """Convert to ChatML format for training."""
    instruction = example.get("instruction") or example.get("question") or example.get("prompt", "")
    inp = example.get("input", "")
    output = example.get("output") or example.get("answer") or example.get("response", "")

    if inp:
        instruction = f"{instruction}\n\n{inp}"

    messages = [
        {"role": "system", "content": PUB_AI_SYSTEM},
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

if CUSTOM_DATA_PATH:
    # Upload custom data
    with open(CUSTOM_DATA_PATH, 'r') as f:
        raw_data = json.load(f)
    dataset = Dataset.from_list(raw_data)
    print(f"Loaded {len(dataset)} custom examples")
elif USE_BUILTIN_DATASET:
    # Use a coding instruction dataset
    dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    print(f"Loaded CodeAlpaca: {len(dataset)} examples")
else:
    print("No dataset selected! Upload data or enable built-in dataset.")

dataset = dataset.map(format_chat)
print(f"\nDataset ready: {len(dataset)} training examples")
print(f"Sample:\n{dataset[0]['text'][:500]}...")

In [ ]:
#@title 4a. Load All Training Datasets (Recommended)
#@markdown Downloads and combines 7 datasets:
#@markdown - 3 reasoning datasets (Opus 4.6 + Claude Opus)
#@markdown - Roblox-Luau-Reasoning (15.3k Luau code + chain-of-thought)
#@markdown - Roblox/luau_corpus (22.6k Luau prompt→completion)
#@markdown - baby-python-and-lua (sampled 2k Lua snippets)
#@markdown - luau_github (sampled 2k Luau files)

USE_COMBINED_DATASET = True #@param {type:"boolean"}
MAX_SAMPLES_PER_RAW = 2000 #@param {type:"integer"}

import hashlib, json, os, re, random
from collections import Counter
from datasets import load_dataset, Dataset

PUB_AI_SYSTEM = """You are Pub AI, a custom-built AI coding assistant. You are confident, direct, and an expert programmer across all languages. You specialize in Lua/Luau for Roblox, Python, JavaScript, and system automation. You give precise, working code with clear explanations. You think step-by-step through complex problems before giving your final answer."""

COMBINED_PATH = "pub_ai_combined_v2.json"

def _hash(text):
    return hashlib.sha256(re.sub(r"\s+", " ", text.strip().lower()).encode()).hexdigest()[:16]

def _asst(thinking, solution):
    t, s = thinking.strip(), solution.strip()
    if t and s: return f"<think>\n{t}\n</think>\n\n{s}"
    if t: return f"<think>\n{t}\n</think>"
    return s

def _valid(user, asst):
    if not user or not asst or len(user.strip()) < 5 or len(asst.strip()) < 10:
        return False
    for pat in [r"cannot solve", r"problem is incomplete", r"as an ai", r"i('m| am) unable to"]:
        if re.search(pat, asst.lower()): return False
    return True

def _msg(user, asst, cat="general", diff="medium", src="unknown"):
    return {"messages": [
        {"role": "system", "content": PUB_AI_SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": asst},
    ], "category": cat, "difficulty": diff, "source": src, "_h": _hash(user)}

# --- Loaders ---

def load_opus(repo_id, tag):
    print(f"  {repo_id}...")
    ds = load_dataset(repo_id, split="train")
    results = []
    for row in ds:
        problem = (row.get("problem") or "").strip()
        thinking = (row.get("thinking") or "").strip()
        solution = (row.get("solution") or "").strip()
        asst = _asst(thinking, solution)
        if not _valid(problem, asst): continue
        results.append(_msg(problem, asst, row.get("category","general"), row.get("difficulty","medium"), tag))
    print(f"    -> {len(results)} valid")
    return results

def load_claude(repo_id, tag):
    print(f"  {repo_id}...")
    ds = load_dataset(repo_id, split="train")
    results = []
    for row in ds:
        msgs = row.get("messages", [])
        if len(msgs) < 2: continue
        user_c, asst_c = "", ""
        for m in msgs:
            if m["role"] == "user": user_c = (m.get("content") or "").strip()
            elif m["role"] == "assistant": asst_c = (m.get("content") or "").strip()
        match = re.search(r"<think>(.*?)</think>(.*)", asst_c, re.DOTALL)
        if match: asst_c = _asst(match.group(1), match.group(2))
        if not _valid(user_c, asst_c): continue
        results.append(_msg(user_c, asst_c, "code", "hard", tag))
    print(f"    -> {len(results)} valid")
    return results

def load_luau_reasoning(max_n=None):
    print(f"  TorpedoSoftware/Roblox-Luau-Reasoning-v1.0...")
    ds = load_dataset("TorpedoSoftware/Roblox-Luau-Reasoning-v1.0", split="train")
    results = []
    for row in ds:
        prompt = (row.get("prompt") or "").strip()
        code = (row.get("code") or "").strip()
        cot = (row.get("chain_of_thought") or "").strip()
        explanation = (row.get("explanation") or "").strip()
        asst = _asst(cot, f"```lua\n{code}\n```\n\n{explanation}" if explanation else f"```lua\n{code}\n```")
        if not _valid(prompt, asst): continue
        results.append(_msg(prompt, asst, "code", "hard", "luau-reasoning"))
    if max_n and len(results) > max_n:
        random.seed(42)
        results = random.sample(results, max_n)
    print(f"    -> {len(results)} valid")
    return results

def load_luau_corpus(max_n=None):
    print(f"  Roblox/luau_corpus...")
    ds = load_dataset("Roblox/luau_corpus", split="train")
    results = []
    for row in ds:
        prompt = (row.get("prompt") or "").strip()
        completion = (row.get("completion") or "").strip()
        if not prompt or not completion: continue
        user = f"Complete this Luau code:\n```lua\n{prompt}\n```"
        asst = f"```lua\n{prompt}{completion}\n```"
        if len(user) > 4000 or len(asst) > 8000: continue
        if not _valid(user, asst): continue
        results.append(_msg(user, asst, "code", "medium", "luau-corpus"))
    if max_n and len(results) > max_n:
        random.seed(42)
        results = random.sample(results, max_n)
    print(f"    -> {len(results)} valid")
    return results

def load_baby_lua(max_n=2000):
    print(f"  nilq/baby-python-and-lua (sampling {max_n})...")
    ds = load_dataset("nilq/baby-python-and-lua", split="train", streaming=True)
    lua_samples = []
    for row in ds:
        content = (row.get("content") or "").strip()
        if not content or len(content) < 50 or len(content) > 4000: continue
        lua_samples.append(content)
        if len(lua_samples) >= max_n * 5: break  # collect extra, then sample
    random.seed(42)
    sampled = random.sample(lua_samples, min(max_n, len(lua_samples)))
    results = []
    for code in sampled:
        first_line = code.split("\n")[0].strip()
        user = f"Explain and improve this Lua code:\n```lua\n{code[:500]}\n```"
        asst = f"Here's the code with analysis:\n\n```lua\n{code}\n```"
        if _valid(user, asst):
            results.append(_msg(user, asst, "code", "medium", "baby-lua"))
    print(f"    -> {len(results)} valid")
    return results

def load_luau_github(max_n=2000):
    print(f"  kefir090/luau_github (sampling {max_n})...")
    ds = load_dataset("kefir090/luau_github", split="train", streaming=True)
    files = []
    for row in ds:
        content = (row.get("file_content") or "").strip()
        path = (row.get("file_path") or "").strip()
        if not content or len(content) < 50 or len(content) > 4000: continue
        files.append((path, content))
        if len(files) >= max_n * 3: break
    random.seed(42)
    sampled = random.sample(files, min(max_n, len(files)))
    results = []
    for path, code in sampled:
        user = f"Explain what this Luau script does:\n```lua\n{code[:800]}\n```"
        asst = f"This script (`{path}`) does the following:\n\n```lua\n{code}\n```"
        if _valid(user, asst):
            results.append(_msg(user, asst, "code", "medium", "luau-github"))
    print(f"    -> {len(results)} valid")
    return results

# --- Main ---

if USE_COMBINED_DATASET:
    if os.path.exists(COMBINED_PATH):
        with open(COMBINED_PATH, "r") as f:
            raw_combined = json.load(f)
        print(f"Loaded cached {len(raw_combined)} examples")
    else:
        print("Building combined dataset...\n")
        all_ex = []

        # Reasoning datasets
        all_ex.extend(load_opus("nohurry/Opus-4.6-Reasoning-3000x-filtered", "opus-3000-filtered"))
        all_ex.extend(load_claude("TeichAI/claude-4.5-opus-high-reasoning-250x", "claude-opus-250"))
        all_ex.extend(load_opus("crownelius/Opus-4.6-Reasoning-3300x", "opus-3300"))

        # Roblox & Luau datasets
        all_ex.extend(load_luau_reasoning(max_n=10000))
        all_ex.extend(load_luau_corpus(max_n=5000))
        all_ex.extend(load_baby_lua(max_n=MAX_SAMPLES_PER_RAW))
        all_ex.extend(load_luau_github(max_n=MAX_SAMPLES_PER_RAW))

        print(f"\n  Total before dedup: {len(all_ex)}")
        seen = set()
        raw_combined = []
        for ex in all_ex:
            h = ex.pop("_h")
            if h not in seen:
                seen.add(h)
                raw_combined.append(ex)
        print(f"  After dedup: {len(raw_combined)}")
        with open(COMBINED_PATH, "w") as f:
            json.dump(raw_combined, f, ensure_ascii=False)

    def format_combined(example):
        msgs = example["messages"]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return {"text": text}

    dataset = Dataset.from_list(raw_combined)
    dataset = dataset.map(format_combined)
    print(f"\nDataset ready: {len(dataset)} training examples")
    sources = Counter(r["source"] for r in raw_combined)
    cats = Counter(r["category"] for r in raw_combined)
    for label, counts in [("Source", sources), ("Category", cats)]:
        print(f"  {label}: {dict(counts)}")
    print(f"\nSample:\n{dataset[0]['text'][:500]}...")
else:
    print("Combined dataset skipped. Use cell 4 for other data sources.")

In [ ]:
#@title 4b. (Optional) Upload Custom Training Data
#@markdown Run this cell to upload a JSON file from your computer
from google.colab import files

print("Upload your training data (JSON file):")
print("Format: [{\"question\": \"...\", \"answer\": \"...\"}]")
print("Or: [{\"instruction\": \"...\", \"output\": \"...\"}]")
print()

uploaded = files.upload()
if uploaded:
    filename = list(uploaded.keys())[0]
    with open(filename, 'r') as f:
        raw_data = json.load(f)
    dataset = Dataset.from_list(raw_data)
    dataset = dataset.map(format_chat)
    print(f"\nLoaded {len(dataset)} custom examples from {filename}")
    print(f"Sample:\n{dataset[0]['text'][:500]}...")

In [ ]:
#@title 4c. (Optional) Paste Training Data Directly
#@markdown Paste Q&A pairs here. One per line, format: Q: ... | A: ...
PASTE_DATA = """Q: How do I make a part in Roblox spin? | A: Use RunService.Heartbeat to rotate the part every frame: game:GetService('RunService').Heartbeat:Connect(function(dt) part.CFrame = part.CFrame * CFrame.Angles(0, math.rad(90) * dt, 0) end)
Q: What is a ModuleScript? | A: A ModuleScript is a reusable piece of code in Roblox. It returns a value (usually a table) when required. Use it to share functions between scripts: local module = {} function module.greet() print('Hello') end return module
Q: How to send an HTTP request in Roblox? | A: Use HttpService:RequestAsync() or HttpService:GetAsync()/PostAsync(). Example: local http = game:GetService('HttpService') local response = http:GetAsync('https://api.example.com/data') print(response)
""" #@param {type:"string"}

if PASTE_DATA.strip():
    pairs = []
    for line in PASTE_DATA.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        if '|' in line and line.startswith('Q:'):
            parts = line.split('|', 1)
            q = parts[0].replace('Q:', '').strip()
            a = parts[1].replace('A:', '').strip()
            pairs.append({"question": q, "answer": a})
    if pairs:
        extra = Dataset.from_list(pairs).map(format_chat)
        from datasets import concatenate_datasets
        dataset = concatenate_datasets([dataset, extra])
        print(f"Added {len(pairs)} pasted examples. Total: {len(dataset)}")
    else:
        print("No valid Q&A pairs found in pasted text.")

In [ ]:
#@title 5. Train! (Takes ~30-60 min on free T4)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting training...")
print(f"Dataset: {len(dataset)} examples")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"LoRA rank: {LORA_RANK}")
print("="*50)

stats = trainer.train()

print("\n" + "="*50)
print(f"Training complete!")
print(f"Total steps: {stats.global_step}")
print(f"Final loss: {stats.training_loss:.4f}")

In [ ]:
#@title 6. Test Your Model
FastLanguageModel.for_inference(model)

test_prompt = "Write a Roblox script that makes a part change color every second" #@param {type:"string"}

messages = [
    {"role": "system", "content": PUB_AI_SYSTEM},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"\nPub AI: {response}")

In [ ]:
#@title 7. Save & Export Model
# Save LoRA adapters
model.save_pretrained(f"{OUTPUT_MODEL_NAME}-lora")
tokenizer.save_pretrained(f"{OUTPUT_MODEL_NAME}-lora")
print(f"LoRA adapters saved to {OUTPUT_MODEL_NAME}-lora/")

# Save merged model (full weights)
model.save_pretrained_merged(f"{OUTPUT_MODEL_NAME}-merged", tokenizer, save_method="merged_16bit")
print(f"Merged model saved to {OUTPUT_MODEL_NAME}-merged/")

if EXPORT_GGUF:
    print("\nExporting to GGUF (for Ollama)...")
    model.save_pretrained_gguf(
        f"{OUTPUT_MODEL_NAME}-gguf",
        tokenizer,
        quantization_method="q4_k_m",  # Good balance of quality and size
    )
    print(f"GGUF exported to {OUTPUT_MODEL_NAME}-gguf/")

if PUSH_TO_HUB and HF_USERNAME:
    from huggingface_hub import login
    login()  # Will prompt for token
    model.push_to_hub_merged(
        f"{HF_USERNAME}/{OUTPUT_MODEL_NAME}",
        tokenizer,
        save_method="merged_16bit",
    )
    if EXPORT_GGUF:
        model.push_to_hub_gguf(
            f"{HF_USERNAME}/{OUTPUT_MODEL_NAME}-GGUF",
            tokenizer,
            quantization_method="q4_k_m",
        )
    print(f"\nPushed to HuggingFace: {HF_USERNAME}/{OUTPUT_MODEL_NAME}")

## Deploy to HuggingFace Inference

After the model is pushed to HuggingFace, set up a **Serverless Inference Endpoint**:

1. Go to `https://huggingface.co/<YOUR_USERNAME>/pub-ai`
2. Click **Deploy** → **Inference Endpoints**
3. Pick **GPU** (T4 is cheapest) or use the **free Serverless API** for small models
4. Your endpoint URL will look like: `https://api-inference.huggingface.co/models/<YOUR_USERNAME>/pub-ai`

Then set these env vars on Railway:
```
VLLM_HOST=https://api-inference.huggingface.co/models/<YOUR_USERNAME>/pub-ai
VLLM_API_KEY=hf_xxxxxxxxxxxx
VLLM_MODEL_NAME=pub-ai
```

Your backend already routes to this automatically via the vLLM provider.

In [ ]:
#@title 8. Download GGUF File
#@markdown Download the GGUF file to use with Ollama locally
import glob
from google.colab import files

gguf_files = glob.glob(f"{OUTPUT_MODEL_NAME}-gguf/*.gguf")
if gguf_files:
    print(f"Found GGUF: {gguf_files[0]}")
    print("Downloading... (this may take a few minutes for large files)")
    files.download(gguf_files[0])
else:
    print("No GGUF file found. Run the export step first.")

## 9. Deploy with Ollama

After downloading the GGUF file, run these commands on your machine:

```bash
# Create a Modelfile
cat > Modelfile << 'EOF'
FROM ./pub-ai-gguf/unsloth.Q4_K_M.gguf

SYSTEM """You are Pub AI, a custom-built AI coding assistant. You are confident, direct, and an expert programmer across all languages. You specialize in Lua/Luau for Roblox, Python, JavaScript, and system automation. You give precise, working code with clear explanations."""

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER num_predict 2048
EOF

# Create the Ollama model
ollama create pub-ai -f Modelfile

# Test it
ollama run pub-ai "Write a Python function to sort a list"
```

Then update your Pub AI backend `.env`:
```
AI_PROVIDER=ollama
OLLAMA_MODEL_NAME=pub-ai
```

In [ ]:
#@title 10. (Optional) DPO Training from Feedback
#@markdown Upload a JSON file with preference pairs:
#@markdown ```json
#@markdown [{"prompt": "...", "chosen": "good response", "rejected": "bad response"}]
#@markdown ```

RUN_DPO = False #@param {type:"boolean"}
DPO_DATA_PATH = "" #@param {type:"string"}
DPO_EPOCHS = 1 #@param {type:"integer"}

if RUN_DPO and DPO_DATA_PATH:
    from trl import DPOTrainer, DPOConfig

    with open(DPO_DATA_PATH, 'r') as f:
        dpo_data = json.load(f)

    dpo_dataset = Dataset.from_list(dpo_data)
    print(f"DPO dataset: {len(dpo_dataset)} preference pairs")

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,  # uses implicit reference
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
        args=DPOConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_train_epochs=DPO_EPOCHS,
            learning_rate=5e-5,
            fp16=True,
            logging_steps=5,
            output_dir="dpo_outputs",
            report_to="none",
        ),
        beta=0.1,
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=MAX_SEQ_LENGTH // 2,
    )

    print("Starting DPO training...")
    dpo_stats = dpo_trainer.train()
    print(f"DPO complete! Loss: {dpo_stats.training_loss:.4f}")

    model.save_pretrained(f"{OUTPUT_MODEL_NAME}-dpo")
    print(f"DPO model saved to {OUTPUT_MODEL_NAME}-dpo/")
elif RUN_DPO:
    print("Upload DPO data first (preference pairs JSON)")
else:
    print("DPO training skipped. Enable RUN_DPO and provide data to train from feedback.")